# Data Pre-processing - custom-methods
---


This notebook filters raw data and flags suspicious values as wrong using different cleaning and quality control techniques


In [ ]:
import os
from pathlib import Path
import requests
import pandas as pd
import numpy as np
import json


In [ ]:

cwd = os.getcwd()
cwd_main = os.path.abspath(os.path.join(cwd, os.pardir))
os.chdir(cwd_main)
print(cwd_main)


import config_tool as cfm


## Define name of city project to work
---


Define the name of the **`city`** (projectname) to upload/save all data during the workflow


In [ ]:

city = os.environ.get("QC_PROJECT_ID", "project_id")


## Import input data of the project - config_project.py
---


Before running, set `QC_DATA_ROOT` or create a local `path_to_data.txt` from `path_to_data.txt.example`; the submitted archive does not contain data files.


In [ ]:

cwd_project = Path(cfm.cwd_data) / city


os.chdir(cwd_project)
print(cwd_project)


import config_project as cfp


In [ ]:

cwd_data_str = cfp.cwd_data_str
cwd_data_meta = cfp.cwd_data_meta
cwd_data_qc = cfp.cwd_data_qc
cwd_data_qc_results  = cfp.cwd_results_qc
cws_data_raw = cfp.cwd_data_raw


print(cwd_main)
print(cwd_project)
print(cwd_data_str)
print(cwd_data_meta)
print(cwd_data_qc)
print(cwd_data_qc_results)
print(cws_data_raw)


In [ ]:

print("city: ",cfp.city)
print("first date: ",cfp.first_date)
print("last date: ",cfp.last_date)
print("lat,long: ",cfp.lat,cfp.long)
print("plot: ",cfp.plot)


start_date = pd.to_datetime(cfp.first_date, format='%d-%m-%Y %H:%M')
end_date = pd.to_datetime(cfp.last_date, format='%d-%m-%Y %H:%M')
print("data from ", start_date, " to ", end_date)


## Read structured data in projectname\data\11_structured   
---


Read data from CWS - Netatmo and Wunderground


In [ ]:



os.chdir(cwd_data_meta)
name_coordinates_wunder=f"Coordinates_{cfp.city}_CWS_Wunderground_str_all.csv"
CWS_coordinates_wunder = pd.read_csv(name_coordinates_wunder)

os.chdir(cwd_data_str)
name_ta_wunder=f"ta_{cfp.city}_{start_date.year}-{end_date.year}_h_CWS_Wunderground_str.csv"
CWS_ta_wunder = pd.read_csv(name_ta_wunder, index_col='date',parse_dates=True)


CWS_coordinates_wunder.info()


In [ ]:


os.chdir(cwd_data_meta)
name_coordinates_net=f"Coordinates_{cfp.city}_CWS_Netatmo_str_all.csv"
CWS_coordinates_net = pd.read_csv(name_coordinates_net)


os.chdir(cwd_data_str)
name_ta_net=f"ta_{cfp.city}_{start_date.year}-{end_date.year}_h_CWS_Netatmo_str.csv"
CWS_ta_net = pd.read_csv(name_ta_net, index_col='date',parse_dates=True)


CWS_ta_net.info()


Read data from OWS (e.g. metoffice) - Omit this step if you don't have data from OWS


In [ ]:


os.chdir(cwd_data_meta)
name_coordinates_ows=f"Coordinates_{cfp.city}_OWS_str_all.csv"
OWS_coordinates = pd.read_csv(name_coordinates_ows,sep=";")

os.chdir(cwd_data_str)
name_ta_ows=f"ta_{cfp.city}_{start_date.year}-{end_date.year}_h_OWS_str.csv"
OWS_ta = pd.read_csv(name_ta_ows, index_col='date',parse_dates=True)


OWS_ta.info()


## Pre-processing
---


In [ ]:
os.chdir(cfm.cwd_scripts_preprocesing)


Update projectdata.json file with project variables


In [ ]:

d = {
     'city':cfp.city,
     'lat':cfp.lat,
     'long':cfp.long,
     'plot':cfp.plot,
     "first_date":cfp.first_date,
     "last_date":cfp.last_date,
     "color_net":cfp.color_net,
     "color_wund":cfp.color_wund,
     "color_ows":cfp.color_ows,
     "color_cws":cfp.color_cws,
    "color_outliers":cfp.color_outliers,
    "cwd_data_str":cfp.cwd_data_str,
    "cwd_data_meta":cfp.cwd_data_meta,
    "cwd_data_qc":cfp.cwd_data_qc,
    "cwd_data_qc_results":cfp.cwd_results_qc,
    "name_coordinates_wunder":name_coordinates_wunder,
    "name_ta_wunder":name_ta_wunder,
    "name_coordinates_net":name_coordinates_net,
    "name_ta_net":name_ta_net,
    "name_coordinates_ows":name_coordinates_ows,
    "name_ta_ows":name_ta_ows
    }

with open('projectdata.json', 'w') as fp:
    json.dump(d, fp)


Run pre-processing


In [ ]:
import runpy

qc_script = Path(cfm.cwd_scripts_preprocesing) / "qc_custom-method.py"
runpy.run_path(str(qc_script), run_name="__main__")


## Statistics before and after pre-processing
---


Check statistics before and after the pre-processing techniques


In [ ]:
os.chdir(cwd_data_qc_results)
os.listdir()


In [ ]:

CWS_stats = pd.read_csv(f"Pre-processing_{city}_2021_CWS_all_G8_g8_Statistics.csv", sep=",")


In [ ]:
CWS_stats


In [ ]:

OWS_stats = pd.read_csv(f"Pre-processing_{city}_2021_OWS_all_G8_g8_Statistics.csv", sep=",")


In [ ]:
OWS_stats
